# Agente simples com ferramentas

Neste exemplo, vamos criar um agente simples que pode responder a perguntas sobre o clima usando uma ferramenta personalizada. A ferramenta simula a obtenção de informações meteorológicas para um local específico.

## Carregando as dependências

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from random import randint
from typing import Annotated

from agent_framework import tool
from pydantic import Field

## Carregando variáveis de ambiente

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

In [ ]:
# print(f"AZURE_OPENAI_ENDPOINT: {os.getenv('AZURE_OPENAI_ENDPOINT')}")
# print(f"AZURE_OPENAI_CHAT_DEPLOYMENT_NAME: {os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME')}")
# print(f"AZURE_OPENAI_API_VERSION: {os.getenv('AZURE_OPENAI_API_VERSION')}")

## Função simples para obter o clima

Esta funcao simula a obtenção de informações meteorológicas para um local específico. Ela retorna uma string com o clima atual para o local fornecido.

In [ ]:
def get_weather_simple(location):
    conditions = ["ensolarado", "nublado", "chuvoso", "tempestuoso"]
    return f"O clima em {location} está {conditions[randint(0, 3)]} com máxima de {randint(10, 30)}°C."

Esta funcao é exatamente igual a anterior mas com a adição do decorador @tool, que é usado para marcar a função como uma ferramenta que pode ser utilizada por agentes. O decorador inclui uma descrição da função, o que ajuda os agentes a entenderem quando e como usar essa ferramenta.

In [ ]:
@tool(approval_mode="never_require")
def get_weather(
    location: Annotated[str, Field(description="O local para obter informações do clima.")],
) -> str:
    """Obter o clima para um local específico."""

    conditions = ["ensolarado", "nublado", "chuvoso", "tempestuoso"]
    return f"O clima em {location} está {conditions[randint(0, 3)]} com máxima de {randint(10, 30)}°C."

## Função para recuperar itens de menu

Esta função simula o menu de um restaurante italiano. Ela utiliza o decorador @tool para marcar a função como uma ferramenta que pode ser utilizada por agentes. O parâmetro categoria é opcional, com valor padrão "massas", permitindo que o agente solicite recomendações de pratos específicos ou obtenha sugestões de massas quando nenhuma categoria é especificada.

In [ ]:
@tool(approval_mode="never_require")
def get_menu(
    categoria: Annotated[str, Field(description="A categoria do menu: massas, carnes ou frutos do mar. Opcional, padrão é massas.")] = "massas",
) -> str:
    """Obter itens do menu italiano para uma categoria específica. Se a categoria não for informada, retorna massas por padrão."""
    menus = {
        "massas": ["Spaghetti Carbonara", "Fettuccine Alfredo", "Penne Arrabbiata", "Lasagna Bolognese", "Ravioli de Ricota"],
        "carnes": ["Osso Buco", "Saltimbocca alla Romana", "Bistecca alla Fiorentina", "Pollo alla Parmigiana", "Vitello Tonnato"],
        "frutos do mar": ["Linguine alle Vongole", "Risotto ai Frutti di Mare", "Branzino al Forno", "Calamari Fritti", "Salmone alla Griglia"]
    }

    categoria_lower = categoria.lower()
    if categoria_lower in menus:
        item = menus[categoria_lower][randint(0, len(menus[categoria_lower]) - 1)]
        preco = randint(35, 85)
        return f"Recomendação de {categoria}: {item} - R$ {preco},00"
    else:
        return f"Desculpe, não temos a categoria '{categoria}' no menu. Temos: massas, carnes ou frutos do mar."

## Criando o cliente

Aqui mostramos algumas maneiras de inicializar o cliente do agente, usando diferentes tipos de chaves de API. O cliente é necessário para que o agente possa acessar as ferramentas e realizar suas tarefas.

In [ ]:
from agent_framework.azure import AzureOpenAIChatClient

# Usando variáveis de ambiente
# Set AZURE_OPENAI_ENDPOINT=""
# Set AZURE_OPENAI_CHAT_DEPLOYMENT_NAME=""
# Set AZURE_OPENAI_API_VERSION=""
# Set AZURE_OPENAI_API_KEY=""
client = AzureOpenAIChatClient()

# # Ou passando os parâmetros diretamente
# client = AzureOpenAIChatClient(
#     endpoint="",
#     deployment_name="",
#     api_key=""
# )

# # Ou carregando a partir de um arquivo .env
# client = AzureOpenAIChatClient(
#     env_file_path="path/to/.env"
# )

## Criando o agente

In [ ]:
agent = client.as_agent(
    instructions="Você é um agente útil de informações sobre o clima.",
    tools=[get_weather,get_menu]
)

## Chamando o agente

In [ ]:
query = "Como está o clima em Sao Paulo?"
print(f"Usuário: {query}")

In [ ]:
result = await agent.run(query)
print(f"Resultado: {result}\n")

## Chamando o agente

In [ ]:
query = "me varias sugestoes de frutos do mar do menu mas sou alergico a camarao"
print(f"Usuário: {query}")

In [ ]:
result = await agent.run(query)
print(f"Resultado: {result}\n")